In [ ]:
# Problema: Resolver las preguntas de Drivers con los mismos datos, usando operaciones clave--valor.

import csv
from collections import defaultdict
from pathlib import Path

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "data").is_dir() and (p / "submission").is_dir())
def load(name):
    with (ROOT / "data" / name).open(encoding="utf-8", newline="") as file:
        return list(csv.DictReader(file))
timesheet, drivers = load("timesheet.csv"), load("drivers.csv")

In [ ]:
# Map: driverId produce medidas parciales; reduce: combina suma, conteo, mínimo y máximo.

partials = defaultdict(lambda: [0.0, 0.0, 0, float("inf"), float("-inf")])
for row in timesheet:
    hours, miles = float(row["hours-logged"]), float(row["miles-logged"])
    value = partials[row["driverId"]]; value[0] += hours; value[1] += miles; value[2] += 1; value[3] = min(value[3], hours); value[4] = max(value[4], hours)
aggregates = {key: {"hours-logged": round(v[0], 2), "miles-logged": round(v[1], 2), "mean_hours-logged": round(v[0] / v[2], 2), "min_hours-logged": round(v[3], 2), "max_hours-logged": round(v[4], 2)} for key, v in partials.items()}

In [ ]:
# La segunda pasada compara cada semana con el promedio reducido; la unión incorpora el nombre por driverId.

names = {row["driverId"]: row["name"] for row in drivers}
summary = [{"driverId": key, "name": names[key], **value} for key, value in aggregates.items()]
summary.sort(key=lambda row: int(row["driverId"]))
below_average = [{**row, "mean_hours-logged": aggregates[row["driverId"]]["mean_hours-logged"]} for row in timesheet if float(row["hours-logged"]) < aggregates[row["driverId"]]["mean_hours-logged"]]
top10 = sorted(summary, key=lambda row: row["miles-logged"], reverse=True)[:10]

In [ ]:
# Se materializan las mismas preguntas que en el PRE de pandas.

for name, result in {"summary.csv": summary, "below_average_hours.csv": below_average, "top10_drivers.csv": top10}.items():
    with (ROOT / "submission" / name).open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(result[0]), lineterminator="\n")
        writer.writeheader(); writer.writerows(result)